# **RAG**

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-google-genai langchain-groq langgraph faiss-cpu pypdf requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 22.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [ ]:
import os
import requests
from google.colab import userdata

# 1. Masukkan API Keys (Bisa lewat Colab Secrets / userdata atau di-assign langsung)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')  # Atau "AIzaSy..."
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')      # Atau "gsk_..."

# 2. URL File Dokumen dari GitHub
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

REPO_NAME = "Sjahid-Akbar-FP-Chatbot"

# Hapus folder lama jika sudah ada (agar selalu fresh)
!rm -rf {REPO_NAME}

# Clone seluruh isi repositori GitHub kamu
!git clone https://github.com/dimasajisaka03-boop/{REPO_NAME}.git

/tmp/ipykernel_919/2945696018.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Cloning into 'Sjahid-Akbar-FP-Chatbot'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 5.00 KiB | 5.00 MiB/s, done.


In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load SEMUA PDF dari folder repositori GitHub
loader = PyPDFDirectoryLoader(REPO_NAME)
docs = loader.load()

print(f"Total halaman PDF yang berhasil dibaca: {len(docs)}")

# 2. Split Teks menjadi Chunks (Persis Notebook Asli)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

# 3. Inisialisasi Embedding Model (Menggunakan nama model dari Notebook Asli)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# 4. Buat Vector Database FAISS dan Retriever
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector database berhasil dibuat!")

Total halaman PDF yang berhasil dibaca: 2
Vector database berhasil dibuat!


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Menggunakan nama model Groq yang stabil
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Prompt Template RAG (Persis seperti notebook asli)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Anda adalah asisten AI yang membantu menjawab pertanyaan berdasarkan dokumen.\n"
               "Gunakan konteks berikut untuk menjawab pertanyaan. "
               "Jika tidak tahu, katakan tidak tahu.\n\n"
               "Konteks:\n{context}"),
    ("human", "{question}")
])

In [ ]:
from typing import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

# State
class GraphState(TypedDict):
    question: str
    context: List[Document]
    answer: str

# Node Retrieve
def retrieve(state: GraphState):
    question = state["question"]
    retrieved_docs = retriever.invoke(question)
    return {"context": retrieved_docs}

# Node Generate
def generate(state: GraphState):
    question = state["question"]
    context_docs = state["context"]

    formatted_context = "\n\n".join(doc.page_content for doc in context_docs)
    chain = prompt | llm
    response = chain.invoke({"context": formatted_context, "question": question})

    return {"answer": response.content}

# Graph Construction
workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
print("LangGraph RAG Siap!")

LangGraph RAG Siap!


In [ ]:
inputs = {"question": "Siapa nama pemilik CV ini dan apa keahliannya?"}
output = app.invoke(inputs)

print("Jawaban:", output["answer"])

Jawaban: Nama pemilik CV ini adalah **Rizky Andika Pratama**.

**Keahlian utama**nya meliputi:

- **Data Science & Machine Learning** – pemodelan prediktif, NLP, deep learning, ensemble methods.  
- **MLOps** – desain dan pengelolaan pipeline end‑to‑end dengan Apache Spark, Airflow, dbt, MLflow, Vertex AI, Docker, Kubernetes, FastAPI.  
- **Basis Data** – PostgreSQL, BigQuery, MongoDB, Redis, Elasticsearch.  
- **Cloud Computing** – Google Cloud Platform (sertifikasi profesional) dan Amazon Web Services (sertifikasi associate).  
- **Visualisasi Data** – Tableau, Metabase, Matplotlib, Seaborn, Plotly.  
- **Domain pengalaman** – e‑commerce (Shopee) dan fintech, termasuk rekomendasi produk real‑time, moderasi ulasan berbasis NLP, serta pembangunan feature store terpusat.


# **Streamlit**

In [ ]:
!pip install -q streamlit pyngrok langchain langchain-community langchain-core langchain-google-genai langchain-groq langgraph faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 46.2 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [ ]:
# ── 2. Sidebar: Pengaturan API Keys & Repo ────────────────────────────────────
import streamlit as st
with st.sidebar:
    st.subheader("Pengaturan")

    # Mengambil otomatis dari Colab Secrets/Environment
    from google.colab import userdata

    try:
        env_google = userdata.get('GOOGLE_API_KEY')
        env_groq = userdata.get('GROQ_API_KEY')
    except:
        env_google = ""
        env_groq = ""

    google_api_key = st.text_input("Google AI API Key", value=env_google, type="password")
    groq_api_key = st.text_input("Groq API Key", value=env_groq, type="password")

    repo_url = st.text_input(
        "URL GitHub Repository",
        value="https://github.com/dimasajisaka03-boop/Sjahid-Akbar-FP-Chatbot.git"
    )

    reset_button = st.button("Reset Percakapan")

2026-09-07 16:49:23.447 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-07 16:49:23.872 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-09-07 16:49:23.875 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-07 16:49:23.877 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-07 16:49:25.347 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-07 16:49:25.348 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-07 16:49:25.350 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-07 16:49:25.352 Session state does not 

In [ ]:
%%writefile app.py
import os
import streamlit as st
from typing import List, TypedDict
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

# ── 1. Konfigurasi Halaman ───────────────────────────────────────────────────
st.title("🤖 Chatbot Sjahid Akbar")
st.caption("Chatbot RAG menggunakan LangChain, LangGraph, FAISS, dan Groq LLM")

# ── 2. Sidebar: Pengaturan API Keys & Repo ────────────────────────────────────
with st.sidebar:
    st.subheader("Pengaturan")
    google_api_key = st.text_input("Google AI API Key", type="password")
    groq_api_key = st.text_input("Groq API Key", type="password")

    repo_url = st.text_input(
        "URL GitHub Repository",
        value="https://github.com/dimasajisaka03-boop/Sjahid-Akbar-FP-Chatbot.git"
    )

    reset_button = st.button("Reset Percakapan", help="Hapus riwayat pesan dan mulai dari awal")

# ── 3. Validasi API Key ──────────────────────────────────────────────────────
if not google_api_key or not groq_api_key:
    st.info("Masukkan Google AI API Key dan Groq API Key di sidebar untuk mulai.", icon="🗝️")
    st.stop()

# Set API Key ke environment
os.environ["GOOGLE_API_KEY"] = google_api_key
os.environ["GROQ_API_KEY"] = groq_api_key

# ── 4. Inisialisasi RAG Pipeline & LangGraph ─────────────────────────────────
if ("rag_app" not in st.session_state) or (getattr(st.session_state, "_last_key", None) != (google_api_key + groq_api_key)):
    try:
        with st.spinner("Memproses repositori GitHub dan membangun Vector Store..."):
            repo_name = repo_url.split("/")[-1].replace(".git", "")

            # Git Clone Repo
            os.system(f"rm -rf {repo_name}")
            os.system(f"git clone {repo_url}")

            # Load PDF dari folder
            loader = PyPDFDirectoryLoader(repo_name)
            docs = loader.load()

            # Text Splitter
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            splits = text_splitter.split_documents(docs)

            # Embedding Gemini
            embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
            vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
            retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

            # Groq LLM
            llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

            # Prompt Template RAG
            prompt = ChatPromptTemplate.from_messages([
                ("system", "Anda adalah asisten AI yang cerdas dan ramah.\n"
                           "Utamakan menjawab pertanyaan menggunakan informasi dari Konteks Dokumen di bawah ini jika relevan.\n"
                           "Jika pertanyaan bersifat umum atau informasinya tidak ditemukan di dalam dokumen, gunakan pengetahuan umum Anda untuk menjawabnya dengan akurat dan jelas.\n\n"
                           "Konteks Dokumen:\n{context}"),
                ("human", "{question}")
            ])

            # LangGraph Definition
            class GraphState(TypedDict):
                question: str
                context: List[Document]
                answer: str

            def retrieve(state: GraphState):
                retrieved_docs = retriever.invoke(state["question"])
                return {"context": retrieved_docs}

            def generate(state: GraphState):
                context_docs = state["context"]
                formatted_context = "\n\n".join(doc.page_content for doc in context_docs)
                chain = prompt | llm
                response = chain.invoke({"context": formatted_context, "question": state["question"]})
                return {"answer": response.content}

            workflow = StateGraph(GraphState)
            workflow.add_node("retrieve", retrieve)
            workflow.add_node("generate", generate)
            workflow.add_edge(START, "retrieve")
            workflow.add_edge("retrieve", "generate")
            workflow.add_edge("generate", END)

            st.session_state.rag_app = workflow.compile()
            st.session_state._last_key = google_api_key + groq_api_key
            st.session_state.pop("messages", None)
            st.success("Basis data dokumen berhasil dibuat!")

    except Exception as e:
        st.error(f"Terjadi kesalahan saat memproses data: {e}")
        st.stop()

# ── 5. Inisialisasi Riwayat Pesan ───────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 6. Tombol Reset ──────────────────────────────────────────────────────────
if reset_button:
    st.session_state.pop("messages", None)
    st.rerun()

# ── 7. Tampilkan Riwayat Percakapan ─────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 8. Input & Respons ───────────────────────────────────────────────────────
prompt_input = st.chat_input("Tanyakan sesuatu tentang dokumen...")

if prompt_input:
    # 1. Simpan dan tampilkan pesan user
    st.session_state.messages.append({"role": "user", "content": prompt_input})
    with st.chat_message("user"):
        st.markdown(prompt_input)

    # 2. Jalankan RAG dan dapatkan respon
    try:
        with st.chat_message("assistant"):
            with st.spinner("Mencari jawaban dari dokumen..."):
                output = st.session_state.rag_app.invoke({"question": prompt_input})
                answer = output["answer"]
                st.markdown(answer)
    except Exception as e:
        answer = f"Terjadi error: {e}"

    # 3. Simpan respon assistant
    st.session_state.messages.append({"role": "assistant", "content": answer})

Overwriting app.py


In [ ]:
proc = run_streamlit("app.py")

Streamlit berjalan: NgrokTunnel: "https://lumpiness-curse-lemon.ngrok-free.dev" -> "http://localhost:8501"
